In [1]:
from hubo_qaoa.utils.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from hubo_qaoa.utils.gfa_utils import gfa_file_to_graph
from qiskit_qaoa.utils.hamiltonian_utils import hamiltonian_to_interactions

from qiskit import QuantumCircuit
from qiskit.circuit.library import CXGate,PauliEvolutionGate
from qiskit.transpiler import PassManager, Layout
from qiskit.transpiler.passes import InverseCancellation, CommutativeCancellation
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping

from qiskit_qaoa.utils.sat_mapper import HigherOrderSatMapper
from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, FindCommutingPauliEvolutionsMulti
# from qiskit_qaoa.utils.commuting_gate_router_precompute_rzz import CommutingGateRouterPrecomputeRzz
from qiskit_qaoa.utils.commuting_gate_router_precompute_rzz_mask import CommutingGateRouterPrecomputeRzz


In [2]:
filename = 'trivial'
copy_numbers = [1.0,1.0,1.0]

In [3]:
filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
graph, n, V, total_weight = gfa_file_to_graph(filepath, copy_numbers)
T = total_weight

num_physical_qubits = n * T

hamiltonian = graph_to_hubo_hamiltonian(graph, n, total_weight, lamda=10, constraint_terms=1.0)
extended_swap_strat = ExtendedSwapStrategy.from_grid(n, T)
program_interactions = hamiltonian_to_interactions(hamiltonian, 0, 1.0)

donor_qc = QuantumCircuit(num_physical_qubits)
qubits = donor_qc.qubits

Keeping constraints at times: [0 1]


In [4]:
pm_rzz = PassManager(
    [
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouterPrecomputeRzz(
            extended_swap_strat,
            max_layers=1,
            perform_extra_swaps=True
        ),
        SwapToFinalMapping(),
        InverseCancellation(gates_to_cancel=[CXGate()]),
        CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)

In [5]:
# mapper = HigherOrderSatMapper(timeout=15)
# sat_results = mapper.hubo_max_sat(
#     num_physical_qubits, program_interactions, extended_swap_strat, 1
# )


# mapping = sat_results[1][1]
# edge_map = dict(mapping)

# layout = Layout({qubits[key]: val for key, val in edge_map.items()})
layout = Layout({
8: qubits[0],
7: qubits[1],
6: qubits[2],
3: qubits[3],
4: qubits[4],
5: qubits[5],
2: qubits[6],
1: qubits[7],
0: qubits[8]
})

qc = QuantumCircuit(num_physical_qubits)
qc.append(PauliEvolutionGate(hamiltonian), [layout.get_virtual_bits()[qubits[i]] for i in range(num_physical_qubits)])     
    
tqc_rzz = pm_rzz.run(qc)   

Max layers needed to apply swap decompose: 1
Gates we cannot directly implement: 16
Transpiling un-implemented gates


In [6]:
tqc_rzz.draw(fold=-1)

global phase: 4.5077
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     ┌───┐            ┌───┐                      ┌───┐                 ┌───┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    ┌───┐                       ┌───┐                                                                                       
q_0 -> 0 ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■───■─────────────────────────────────────────────────────────■────────────────────────■──────────────────────────────────────■────────────■──────────────────────────────────────────────────────────────────────────────────■──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■──────────────────────■─────────────────────────┤ X ├─■──────────┤ X ├──────■───────────────┤ X ├─■───────────────┤ X ├──────────X────────────────────────────────────────────────────────────────────────────────■───────────────────────────────────────■─────────────────────────────────────────────────────────────────────────────────────────■───────────────────────────────────────────────────■──────────────────────────■───X──X────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤ X ├──■─────────────────■──┤ X ├───────────────────────────────────────────────────────────────────────────────────────
                                                                                                                                                                                                                                             ┌─┴─┐ │                                                   ┌───┐ │ZZ(0.625)             ┌─┴─┐                 ┌───┐┌───────────┐ │ZZ(0.625)   │                                                                                  │       ┌───┐┌───┐                          